In [0]:
import requests
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.functions import col, expr, current_timestamp

REDPANDA_BOOTSTRAP = "d8rtvni0pa8fqqs9v5n0.any.us-east-1.mpx.prd.cloud.redpanda.com:9092"
SCHEMA_REGISTRY_URL = "https://d8rtvni0pa8fqqs9v5n0.registry.us-east-1.mpx.prd.cloud.redpanda.com"
REDPANDA_USERNAME = "ecommerce-producer"
REDPANDA_PASSWORD = dbutils.secrets.get(scope="redpanda", key="password")  # see note below

JAAS_CONFIG = (
    f'kafkashaded.org.apache.kafka.common.security.scram.ScramLoginModule required '
    f'username="{REDPANDA_USERNAME}" password="{REDPANDA_PASSWORD}";'
)

def get_schema(subject: str) -> str:
    resp = requests.get(
        f"{SCHEMA_REGISTRY_URL}/subjects/{subject}/versions/latest",
        auth=(REDPANDA_USERNAME, REDPANDA_PASSWORD)
    )
    return resp.json()["schema"]

def read_topic_stream(topic: str):
    return spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", REDPANDA_BOOTSTRAP) \
        .option("kafka.security.protocol", "SASL_SSL") \
        .option("kafka.sasl.mechanism", "SCRAM-SHA-256") \
        .option("kafka.sasl.jaas.config", JAAS_CONFIG) \
        .option("subscribe", topic) \
        .option("startingOffsets", "earliest") \
        .load()

def decode_avro(df, schema_str: str):
    return df.withColumn(
        "avro_payload", expr("substring(value, 6, length(value) - 5)")
    ).withColumn(
        "decoded", from_avro(col("avro_payload"), schema_str)
    ).select(
        "decoded.*",
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp")
    ).withColumn("ingested_at", current_timestamp())

In [0]:
user_activity_schema = get_schema("user-activity-value")

user_activity_stream = read_topic_stream("user-activity")
user_activity_decoded = decode_avro(user_activity_stream, user_activity_schema)

CHECKPOINT_USER_ACTIVITY = "/Volumes/workspace/default/ecommerce_data/checkpoints/bronze_user_activity"

query_user_activity = user_activity_decoded.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_USER_ACTIVITY) \
    .trigger(availableNow=True) \
    .toTable("workspace.default.bronze_user_activity")

query_user_activity.awaitTermination()
print(f"Bronze user_activity written. Rows: {query_user_activity.lastProgress['sink']['numOutputRows'] if query_user_activity.lastProgress else 'N/A'}")

In [0]:
print(f"Total rows in Bronze user_activity: {spark.table('workspace.default.bronze_user_activity').count()}")
spark.table("workspace.default.bronze_user_activity").show(10, truncate=False)